In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# File paths
TRAIN_PATH = Path("corpus/train.jsonl")
TEST_PATH = Path("corpus/test.jsonl")
VALIDATION_PATH = Path("corpus/validation.jsonl")

# Output directory
OUTPUT_DIR = Path("corpus/bertimbau_large_binary_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

# Labels
# 0 = non-pun
# 1 = pun
id2label = {
    0: "0",
    1: "1"
}

label2id = {
    "0": 0,
    "1": 1
}

# Hyperparameters
SEED = 40
MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def read_jsonl(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


train_df = read_jsonl(TRAIN_PATH)
validation_df = read_jsonl(VALIDATION_PATH)
test_df = read_jsonl(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(validation_df.head())
display(test_df.head())

In [ ]:
required_columns = {"id", "text", "label"}

missing_train_columns = required_columns - set(train_df.columns)
missing_validation_columns = required_columns - set(validation_df.columns)
missing_test_columns = required_columns - set(test_df.columns)

if missing_train_columns:
    raise ValueError(f"Missing columns in train file: {missing_train_columns}")

if missing_validation_columns:
    raise ValueError(f"Missing columns in validation file: {missing_validation_columns}")

if missing_test_columns:
    raise ValueError(f"Missing columns in test file: {missing_test_columns}")

train_df["label"] = train_df["label"].astype(int)
validation_df["label"] = validation_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

valid_labels = {0, 1}

if set(train_df["label"].unique()) - valid_labels:
    raise ValueError("Train labels must be only 0 and 1.")

if set(validation_df["label"].unique()) - valid_labels:
    raise ValueError("Validation labels must be only 0 and 1.")

if set(test_df["label"].unique()) - valid_labels:
    raise ValueError("Test labels must be only 0 and 1.")

print("Train label distribution:")
display(train_df["label"].value_counts().sort_index())

print("Validation label distribution:")
display(validation_df["label"].value_counts().sort_index())

print("Test label distribution:")
display(test_df["label"].value_counts().sort_index())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class PunDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.ids = dataframe["id"].astype(str).tolist()
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[index], dtype=torch.long)
        }

In [ ]:
train_dataset = PunDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

validation_dataset = PunDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = PunDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision_macro = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    precision_weighted = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall_weighted = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1_weighted = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,

    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_strategy="epoch",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    save_total_limit=1,

    report_to="none",
    fp16=torch.cuda.is_available(),

    seed=SEED,
    data_seed=SEED
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ]
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("=== Test results ===")

for metric_name, metric_value in test_results.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

In [ ]:
predictions_output = trainer.predict(test_dataset)

y_true = predictions_output.label_ids
y_pred = np.argmax(predictions_output.predictions, axis=1)

print("=== Classification report ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["0", "1"],
        digits=2,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(cm_df)